# N7-11 / N7-10-S2 scoring worker (Colab / Kaggle T4)

Scores one **shard** of `configs/n7_11_jobs.json` (300 DataDecide final checkpoints + 50 PolyPythia
checkpoints) on SciQ, LAMBADA-openai and BLiMP, writing per-question parquet files to Google Drive.

* Open one copy of this notebook **per shard** (e.g. 4 notebooks: `SHARD = 0, 1, 2, 3`, `NSHARDS = 4`).
* Runtime -> Change runtime type -> **T4 GPU**.
* **Resumable**: if Colab disconnects, just run all cells again; finished jobs are skipped.
* Precision is fp16 (validated against DataDecide's released log-likelihoods; see `src/pdt/eval/README.md`).
* When all shards are done, download the Drive folder `pdt_n7_11/` and put it at
  `data/n7_11_outputs/` in the repo; the analysis runs locally.

In [ ]:
SHARD = 0  # 0 .. NSHARDS-1, different in each notebook
NSHARDS = 4
KINDS = "datadecide,hf"  # "datadecide" = N7-11, "hf" = PolyPythia (N7-10 S2)
BRANCH = "phase7/rescore"

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/pdt_n7_11"

In [ ]:
# The repository is private: paste a GitHub token with read access (it is not printed or saved).
import getpass
import os
import subprocess

if not os.path.exists("/content/pdt"):
    token = getpass.getpass("GitHub token: ")
    url = f"https://{token}@github.com/ParthAgarwalGit/pretraining-data-decision-theory.git"
    subprocess.run(["git", "clone", "--quiet", "--branch", BRANCH, url, "/content/pdt"], check=True)
    del token, url
%cd /content/pdt
!git log --oneline -1

In [ ]:
!pip install -q datasets pyarrow safetensors
import torch
import transformers

print(torch.__version__, transformers.__version__, torch.cuda.get_device_name())

In [ ]:
ARGS = f"--shard {SHARD} --nshards {NSHARDS} --kinds {KINDS} --out {OUT}"
!PYTHONPATH=src python experiments/n7_11_score.py {ARGS}

In [ ]:
# Progress over all shards (reads the shared Drive folder).
import glob
import json

done = glob.glob(f"{OUT}/*/done.json")
spec = json.load(open("configs/n7_11_jobs.json"))
print(f"{len(done)} / {spec['n_jobs']} jobs finished")

**Kaggle instead of Colab:** skip the Drive cell, set `OUT = '/kaggle/working/pdt_n7_11'`, add the GitHub token
as a Kaggle secret (or upload the repo as a dataset), and download `/kaggle/working/pdt_n7_11` when done.